# Vision-Based Defect Detection: Dataset Exploration & Validation
### Manufacturing Quality Inspection Pipeline
This notebook explores the dataset generation, statistical distributions, pixel-level defect masks, and environmental perturbations (lighting, orientation, and surface background conditions).

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Ensure project root is accessible
sys.path.append('..')
from src.config import DEFECT_CLASSES, DEFECT_COLORS, DATA_PROCESSED_DIR
from src.synthetic_generator import generate_synthetic_sample, SyntheticSurfaceGenerator
from src.prepare_dataset import prepare_synthetic_dataset, compute_and_print_dataset_stats
from src.inspect import overlay_defect_localization

## 1. Manufacturing Substrates & Defect Simulation
Visualizing normal pristine surfaces alongside the 6 manufacturing defect categories with ground truth localization masks.

In [ ]:
fig, axes = plt.subplots(len(DEFECT_CLASSES), 3, figsize=(12, 2.8 * len(DEFECT_CLASSES)))

for idx, defect_type in enumerate(DEFECT_CLASSES):
    img, mask, info = generate_synthetic_sample(width=256, height=256, defect_type=defect_type)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Overlaid localization
    overlaid = overlay_defect_localization(
        img, mask, defect_type, confidence=0.98, bbox=tuple(info['bbox'])
    )
    overlaid_rgb = cv2.cvtColor(overlaid, cv2.COLOR_BGR2RGB)
    
    # 1. Raw
    axes[idx, 0].imshow(img_rgb)
    axes[idx, 0].set_title(f"{defect_type.upper()} (Raw Image)", fontsize=11, fontweight='bold')
    axes[idx, 0].axis('off')
    
    # 2. Mask
    axes[idx, 1].imshow(mask, cmap='gray')
    axes[idx, 1].set_title(f"Ground Truth Mask ({info['pixel_count']} px)", fontsize=11)
    axes[idx, 1].axis('off')
    
    # 3. Localized BBox & Contour Overlay
    axes[idx, 2].imshow(overlaid_rgb)
    axes[idx, 2].set_title("Localized Inspection HUD", fontsize=11)
    axes[idx, 2].axis('off')

plt.tight_layout()
plt.show()

## 2. Dataset Generation and Statistical Summary
Generate a balanced training batch and compute statistical properties.

In [ ]:
# Generate balanced dataset (20 per class for interactive exploration)
records = prepare_synthetic_dataset(output_dir=DATA_PROCESSED_DIR, num_samples_per_class=20, img_size=(256, 256))
stats = compute_and_print_dataset_stats(records)